# Proyecto intermedio

## Parte 1. Implementacion funcion simular

### Simulación de fragmentación y coalescencia

Tarea: CU-1dffqejt12n

Rama: `CU-1dffqejt12n_Pre-FuncionSimular_Manuel-Enrique-Marin-Cubillos`

Adaptación de `Parte1-Codigo/Parte1.py`, sin gráficas. Requiere Python 3 y NumPy en el entorno de JupyterLab. Ejecuta la celda de definición y luego llama a `simular` con los parámetros de tu experimento.


In [ ]:
import secrets
import time
from numbers import Integral, Real

import numpy as np


def simular(N, nu, pasos, semilla=None):
    """Simula la fragmentación y coalescencia de una fuerza total constante.

    Parámetros
    ----------
    N : int positivo
        Fuerza total. Inicialmente hay N grupos de tamaño 1.
    nu : float entre 0 y 1, inclusive
        Probabilidad de fragmentar el grupo del individuo seleccionado.
        En caso contrario se intenta unirlo al grupo de otro individuo.
    pasos : int no negativo
        Número de iteraciones; cero devuelve la configuración inicial.
    semilla : int no negativo o None
        Si se omite, se genera una semilla aleatoria de 128 bits y se devuelve
        para poder repetir el experimento. No se modifica el generador global.

    Retorna
    -------
    dict
        parametros: N, nu y pasos utilizados.
        semilla: semilla efectiva, incluida cuando se genera automáticamente.
        tiempo_ejecucion_segundos: tiempo desde la inicialización del generador
            hasta el cálculo de la distribución (excluye validación).
        tamanos_grupos: lista de tamaños finales, ordenada por ID interno.
        cantidad_grupos: número total de grupos activos.
        distribucion_tamanos: diccionario tamaño -> {cantidad, frecuencia_relativa}.
            La frecuencia relativa es cantidad / cantidad_grupos.

    Notas
    -----
    Conserva las reglas de Parte1.py: selección uniforme de individuos,
    fragmentación en grupos unitarios y unión de grupos completos. Cada
    individuo tiene la misma probabilidad de selección, por lo que los grupos
    se seleccionan proporcionalmente a su tamaño. No produce gráficas ni imprime.
    La misma semilla y parámetros reproducen los tamaños con la misma versión
    de NumPy; el tiempo medido puede variar. Se usa default_rng, cuya secuencia
    difiere de np.random del script original. El coste es O(N * pasos) en el
    peor caso; ejecutar pasos iteraciones no garantiza estado estacionario.
    """
    for nombre, valor, minimo in (("N", N, 1), ("pasos", pasos, 0)):
        if isinstance(valor, (bool, np.bool_)) or not isinstance(valor, Integral):
            raise TypeError(f"{nombre} debe ser un entero.")
        if valor < minimo:
            raise ValueError(f"{nombre} debe ser mayor o igual que {minimo}.")
    if isinstance(nu, (bool, np.bool_)) or not isinstance(nu, Real):
        raise TypeError("nu debe ser un número real.")
    if not np.isfinite(nu) or not 0 <= nu <= 1:
        raise ValueError("nu debe estar entre 0 y 1 y ser finito.")
    if semilla is not None:
        if isinstance(semilla, (bool, np.bool_)) or not isinstance(semilla, Integral):
            raise TypeError("semilla debe ser un entero o None.")
        if semilla < 0:
            raise ValueError("semilla debe ser no negativa.")

    N, nu, pasos = int(N), float(nu), int(pasos)
    semilla = secrets.randbits(128) if semilla is None else int(semilla)
    inicio = time.perf_counter()
    rng = np.random.default_rng(semilla)
    dueno = np.arange(N, dtype=np.int64)
    siguiente_id_libre = N

    for _ in range(pasos):
        p = rng.integers(0, N)
        g_i = dueno[p]
        if rng.random() < nu:
            miembros = np.where(dueno == g_i)[0]
            num_miembros = len(miembros)
            if num_miembros > 1:
                dueno[miembros] = np.arange(
                    siguiente_id_libre, siguiente_id_libre + num_miembros
                )
                siguiente_id_libre += num_miembros
        else:
            q = rng.integers(0, N)
            g_k = dueno[q]
            if g_i != g_k:
                dueno[dueno == g_k] = g_i

    _, tamanos = np.unique(dueno, return_counts=True)
    valores, conteos = np.unique(tamanos, return_counts=True)
    cantidad_grupos = len(tamanos)
    tamanos_grupos = tamanos.tolist()
    distribucion = {
        int(tamano): {
            "cantidad": int(conteo),
            "frecuencia_relativa": float(conteo / cantidad_grupos),
        }
        for tamano, conteo in zip(valores, conteos)
    }
    tiempo = time.perf_counter() - inicio
    return {
        "parametros": {"N": N, "nu": nu, "pasos": pasos},
        "semilla": semilla,
        "tiempo_ejecucion_segundos": tiempo,
        "tamanos_grupos": tamanos_grupos,
        "cantidad_grupos": cantidad_grupos,
        "distribucion_tamanos": distribucion,
    }


### Experimento reproducible

El ejemplo usa valores pequeños para ejecutarse rápidamente. Para los parámetros originales: `simular(N=10000, nu=0.01, pasos=1000000)`. Esta ejecución puede tardar considerablemente más.


In [ ]:
resultado = simular(N=100, nu=0.01, pasos=1000, semilla=42)
resultado


### Semilla automática y repetición

Guarda la semilla devuelta junto con los parámetros y la versión de NumPy. La distribución cuenta grupos, no individuos; sus frecuencias relativas suman 1. La suma de todos los tamaños siempre es N. Los identificadores internos no se exponen como posiciones persistentes entre experimentos.


In [ ]:
aleatorio = simular(N=100, nu=0.1, pasos=1000)
repetido = simular(**aleatorio['parametros'], semilla=aleatorio['semilla'])
assert aleatorio['tamanos_grupos'] == repetido['tamanos_grupos']
assert sum(aleatorio['tamanos_grupos']) == aleatorio['parametros']['N']
aleatorio['semilla'], aleatorio['distribucion_tamanos']


## Parte 2. Numero de pasos eficaces

## Parte3. Analisis resultados

### Gráfica 1: Distribución en Estado Estacionario (Ley de Potencias)
A continuación se simula el sistema para obtener la distribución de tamaños de los grupos ($n_s$ frente a $s$) y ajustar la regresión lineal que permite estimar el exponente $\alpha$.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def estimar_alpha_con_rango(tamanos, cantidades):
    ts = np.array(tamanos, dtype=float)
    ns = np.array(cantidades, dtype=float)
    min_s = 2
    max_s = max(ts) // 3 if max(ts) > 10 else max(ts)
    mask = (ts >= min_s) & (ts <= max_s) & (ns > 0)
    if np.sum(mask) < 2:
        mask = (ts > 0) & (ns > 0)
    log_s = np.log(ts[mask])
    log_ns = np.log(ns[mask])
    pendiente, intercepto = np.polyfit(log_s, log_ns, 1)
    return -pendiente, ts[mask], ns[mask], pendiente, intercepto

res_principal = simular(N=10000, nu=0.01, pasos=500000, semilla=42)
tamanos_unicos = sorted(res_principal["distribucion_tamanos"].keys())
cantidades_grupos = [res_principal["distribucion_tamanos"][t]["cantidad"] for t in tamanos_unicos]

alpha_val, ts_fit, ns_fit, m, c = estimar_alpha_con_rango(tamanos_unicos, cantidades_grupos)
print(f"Exponente alpha estimado: {alpha_val:.3f}")

plt.figure(figsize=(9, 6))
plt.loglog(tamanos_unicos, cantidades_grupos, 'bo', alpha=0.7, label='Datos de la simulación')
s_linea = np.linspace(min(ts_fit), max(ts_fit), 100)
n_linea = np.exp(c) * (s_linea ** (-alpha_val))
plt.loglog(s_linea, n_linea, 'r-', linewidth=2, label=f'Regresion lineal (alfa = {alpha_val:.2f})')
n_teorica = np.exp(c) * (s_linea ** (-2.5))
plt.loglog(s_linea, n_teorica, 'k--', linewidth=1.5, label='Valor teorico (alfa = 2.5)')
plt.xlabel('Tamano de la unidad (s)', fontsize=12)
plt.ylabel('Numero de grupos (ns)', fontsize=12)
plt.title('Distribucion en Estado Estacionario (N=10000, nu=0.01)', fontsize=13)
plt.legend()
plt.grid(True, which="both", ls="--", alpha=0.5)
plt.show()

### Gráfica 2: Sensibilidad del Exponente $\alpha$ frente a $\nu$
Se realiza un barrido variando el parámetro de fragmentación $\nu$ utilizando múltiples semillas para evaluar la estabilidad y variabilidad de los resultados obtenidos.

In [ ]:
valores_nu = [0.001, 0.005, 0.01, 0.02, 0.05, 0.1]
semillas = [101, 202, 303]
resultados_nu = {}

for nu in valores_nu:
    alphas_semillas = []
    for sem in semillas:
        res_nu = simular(N=5000, nu=nu, pasos=200000, semilla=sem)
        ts_nu = sorted(res_nu["distribucion_tamanos"].keys())
        ns_nu = [res_nu["distribucion_tamanos"][t]["cantidad"] for t in ts_nu]
        a_est, _, _, _, _ = estimar_alpha_con_rango(ts_nu, ns_nu)
        if not np.isnan(a_est):
            alphas_semillas.append(a_est)
    resultados_nu[nu] = {
        "media": np.mean(alphas_semillas) if alphas_semillas else 0,
        "std": np.std(alphas_semillas) if alphas_semillas else 0
    }

nus = list(resultados_nu.keys())
medias_alpha = [resultados_nu[nu]["media"] for nu in nus]
stds_alpha = [resultados_nu[nu]["std"] for nu in nus]

plt.figure(figsize=(8, 5))
plt.errorbar(nus, medias_alpha, yerr=stds_alpha, fmt='o-', color='darkgreen', ecolor='gray', 
             capsize=5, elinewidth=1.5, linewidth=2, label='alfa estimado (Media ± Desv. Est.)')
plt.xlabel('Parametro de fragmentacion (nu)', fontsize=12)
plt.ylabel('Exponente alfa estimado', fontsize=12)
plt.title('Sensibilidad del exponente alfa frente a nu', fontsize=13)
plt.legend()
plt.grid(True, linestyle="--", alpha=0.5)
plt.show()